# Hotel Booking Cancellation & Overbooking Decision
## 05 — Feature Engineering, Train/Test Split and Preprocessing

Prepare the cleaned data for ML without fitting preprocessing on the test set.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

RANDOM_STATE=42

## 2. Load Cleaned Data

In [ ]:
DATA_PATH='../data/processed/hotel_bookings_cleaned.csv'
df=pd.read_csv(DATA_PATH)
print('Shape:',df.shape)
df.head()

## 3. Confirm Leakage Columns Are Absent

In [ ]:
leakage_cols=['reservation_status','reservation_status_date']
print([x for x in leakage_cols if x in df.columns])

## 4. Feature Engineering

Create `total_nights`, `total_guests`, `total_previous_bookings`, `has_special_request`, and `room_type_changed`.

In [ ]:
df_model=df.copy()
df_model['total_nights']=df_model['stays_in_weekend_nights']+df_model['stays_in_week_nights']
df_model['total_guests']=df_model['adults']+df_model['children']+df_model['babies']
df_model['total_previous_bookings']=df_model['previous_cancellations']+df_model['previous_bookings_not_canceled']
df_model['has_special_request']=(df_model['total_of_special_requests']>0).astype(int)
df_model['room_type_changed']=(df_model['reserved_room_type']!=df_model['assigned_room_type']).astype(int)
df_model[['total_nights','total_guests','total_previous_bookings','has_special_request','room_type_changed']].head()

## 5. Remove Redundant Source Columns

In [ ]:
redundant=['stays_in_weekend_nights','stays_in_week_nights','adults','children','babies']
df_model=df_model.drop(columns=redundant)
print(df_model.shape)

## 6. Define X and y

In [ ]:
TARGET='is_canceled'
X=df_model.drop(columns=[TARGET])
y=df_model[TARGET]
print('X:',X.shape,'y:',y.shape)
print((y.value_counts(normalize=True)*100).round(2))

## 7. Stratified Train/Test Split

Split before fitting preprocessing to prevent test-set leakage.

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=RANDOM_STATE,stratify=y)
print('X_train:',X_train.shape)
print('X_test:',X_test.shape)
print('Train cancellation %:',round(y_train.mean()*100,2))
print('Test cancellation %:',round(y_test.mean()*100,2))

## 8. Numerical and Categorical Features

In [ ]:
numerical_features=X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features=X_train.select_dtypes(include=['object','category']).columns.tolist()
print('Numerical:',len(numerical_features),numerical_features)
print('\nCategorical:',len(categorical_features),categorical_features)

## 9. Numerical Pipeline

Median imputation plus standardization.

In [ ]:
numeric_pipeline=Pipeline(steps=[('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())])

## 10. Categorical Pipeline

Most-frequent imputation plus one-hot encoding. Unknown test categories are ignored.

In [ ]:
categorical_pipeline=Pipeline(steps=[('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))])

## 11. Combined Preprocessor

In [ ]:
preprocessor=ColumnTransformer(transformers=[('num',numeric_pipeline,numerical_features),('cat',categorical_pipeline,categorical_features)],remainder='drop')
preprocessor

## 12. Fit Only on Training Data

In [ ]:
X_train_processed=preprocessor.fit_transform(X_train)
X_test_processed=preprocessor.transform(X_test)
print('Processed train:',X_train_processed.shape)
print('Processed test:',X_test_processed.shape)

## 13. Generated Feature Names

In [ ]:
feature_names=preprocessor.get_feature_names_out()
print('Total processed features:',len(feature_names))
print(feature_names[:30])

## 14. Validation

In [ ]:
print('Train rows preserved:',X_train_processed.shape[0]==X_train.shape[0])
print('Test rows preserved:',X_test_processed.shape[0]==X_test.shape[0])
print('Same processed columns:',X_train_processed.shape[1]==X_test_processed.shape[1])
print('Target excluded:',TARGET not in X.columns)

## Modelling Note

In the next notebook, the preprocessor should be placed inside each model `Pipeline`. This keeps preprocessing leakage-safe during model fitting and cross-validation.

## Summary

Completed feature engineering, stratified train/test splitting, numerical/categorical identification, one-hot encoding and scaling pipelines, and leakage-safe preprocessing verification.

### Next Step
Train Logistic Regression, KNN, Decision Tree, Random Forest and a boosting model, then compare Accuracy, Precision, Recall, F1, ROC-AUC and confusion matrices.